In [1]:
# Celda 2: Título del cuaderno / mensaje principal
def banner_title(text):
    line = "=" * (len(text) + 8)
    print(f"\n{line}\n***  {text}  ***\n{line}\n")

banner_title("Agente Municipal — Panel de Revisión de Reportes")
print("Instrucciones:")
print("- Ejecutá las celdas en orden.")
print("- El agente solo podrá ver y actuar sobre reportes de su municipio.")
print("- Si hay errores de importación con dao, avisame y adapto la importación.")



***  Agente Municipal — Panel de Revisión de Reportes  ***

Instrucciones:
- Ejecutá las celdas en orden.
- El agente solo podrá ver y actuar sobre reportes de su municipio.
- Si hay errores de importación con dao, avisame y adapto la importación.


In [2]:
# Celda 1: Inicializar entorno y conectar usando dao.mongo_dao.MongoDAO
# Ejecutá esta celda primero en prueba_agente_municipal_dao.ipynb

from bson import ObjectId
from dao.mongo_dao import MongoDAO
from dao.agente_municipal_dao import AgenteMunicipalDAO

# Instanciar y conectar
mongo_dao = MongoDAO()          # usa config.mongo_uri() y MONGO_DB_NAME del proyecto
db = mongo_dao.connect()        # realiza ping y ensure_indexes (maneja warnings)

# Exponer objetos útiles en el notebook
mongo = mongo_dao               # el DAO expone .db y .client
print("Conectado a DB:", getattr(mongo.db, "name", None))

# Ejemplo de instanciación del DAO del agente (comentar hasta reemplazar ObjectId real)
# agente_municipio_id = ObjectId("6a21d09b4076ce51229df8a4")  # reemplazar por id real
# agente_dao = AgenteMunicipalDAO(mongo, agente_municipio_id=agente_municipio_id)
# agente_dao.ensure_indexes()

# Nota: dejé la creación del AgenteMunicipalDAO comentada para que pegues el municipio correcto
# y no haya efectos no deseados. Ejecutá la celda y avisame para preparar la Celda 2 (título).


Conectado a DB: civictech


In [6]:
# Celda siguiente (ejecutá solo esta celda): mostrar reportes por agente (cada agente por separado)
# Usa 'mongo' ya inicializado (MongoDAO) y el módulo dao.agente_municipal_dao

from bson import ObjectId
from dao.agente_municipal_dao import AgenteMunicipalDAO

def banner(msg):
    line = "=" * (len(msg) + 8)
    print(f"\n{line}\n***  {msg}  ***\n{line}\n")

# Lista de agentes detectados (reemplazá/ajustá si querés)
agentes = [
    {"_id": ObjectId("6a21d09b4076ce51229df8ca"), "nombre": "Inspector Chilecito", "municipio_id": ObjectId("6a21d09b4076ce51229df8a4")},
    {"_id": ObjectId("6a21d09b4076ce51229df8cb"), "nombre": "Inspector La Rioja", "municipio_id": ObjectId("6a21d09b4076ce51229df8a5")},
    {"_id": ObjectId("6a21d09b4076ce51229df8cc"), "nombre": "Inspector Famatina", "municipio_id": ObjectId("6a21d09b4076ce51229df8a3")},
    {"_id": ObjectId("6a21ef2d5669d2ea4c5f3d5c"), "nombre": "Prueba Agente Municipal", "municipio_id": ObjectId("6a21d09b4076ce51229df8a3")},
]

# Iterar por cada agente e imprimir sus reportes (solo los de su municipio)
for ag in agentes:
    try:
        dao = AgenteMunicipalDAO(mongo, agente_municipio_id=ag["municipio_id"])
    except Exception as e:
        print(f"ERROR al instanciar DAO para agente {ag['nombre']}: {type(e).__name__} {e}")
        continue

    # No interrumpe si ensure_indexes falla por permisos
    try:
        dao.ensure_indexes()
    except Exception:
        pass

    rows = []
    try:
        rows = dao.list_reports_by_municipio(limit=500)
    except Exception as e:
        print(f"ERROR al listar reportes para {ag['nombre']}: {type(e).__name__} {e}")
        continue

    banner(f"Agente: {ag['nombre']}  |  municipio_id: {ag['municipio_id']}  |  reportes: {len(rows)}")
    if not rows:
        print("  (No hay reportes visibles para este agente)\n")
        continue

    # Mostrar resumen compacto de cada reporte
    for i, r in enumerate(rows, start=1):
        fecha = r.get("fechaHora_server")
        patente = r.get("patente_vehiculo") or "-"
        estado = r.get("estado") or "-"
        mun_nombre = r.get("municipio_nombre") or "-"
        print(f"{i:03d}. _id: {r.get('_id')} | patente: {patente} | estado: {estado} | fecha: {fecha} | municipio: {mun_nombre}")
    print("\n")  # separación entre agentes



***  Agente: Inspector Chilecito  |  municipio_id: 6a21d09b4076ce51229df8a4  |  reportes: 14  ***

001. _id: 6a236e3202b1dea021ff1130 | patente: EVID-002 | estado: Pendiente | fecha: 2026-06-06 00:47:46.243000 | municipio: Chilecito
002. _id: 6a236be702b1dea021ff112e | patente: EVID-002 | estado: Pendiente | fecha: 2026-06-06 00:37:59.830000 | municipio: Chilecito
003. _id: 6a236a7002b1dea021ff1126 | patente: DEMO123 | estado: Pendiente | fecha: 2026-06-06 00:31:44.742000 | municipio: Chilecito
004. _id: 6a236a4d02b1dea021ff1124 | patente: DEMO123 | estado: Pendiente | fecha: 2026-06-06 00:31:09.276000 | municipio: Chilecito
005. _id: 6a2364aca6caeae53cca07bc | patente: DEMO123 | estado: Pendiente | fecha: 2026-06-06 00:07:08.356000 | municipio: Chilecito
006. _id: 6a235e43a59acb32d36d37eb | patente: DEMO123 | estado: Pendiente | fecha: 2026-06-05 23:39:47.292000 | municipio: Chilecito
007. _id: 6a235d81a59acb32d36d37e9 | patente: DEMO123 | estado: Pendiente | fecha: 2026-06-05 23:36: